## 📑 Core Modifications Production IRT Action Codes 
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/AQB4H)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()

import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

In [0]:
import boto3
import os
 
#takes the values d for dev, qa for test and p for prod depending in which workspace the init variables script is running
# env = os.environ["S3PREFIX"]
 
dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

som_date = spark.sql(
    f'''
    Select MAX(CalendarDate)  SOM
    from `{env_prefix}-cdh`.reference.ref_calendar
    WHERE LastDateOfMonth = last_day(add_months({run_date!r},-1)) and IsBusinessDay = 1
    '''
).collect()[0]['SOM']

###### Schema of the table

In [0]:
table_name = 'Core_ModificationsProductionIRTActionCodes'

schema = (
"BusinessDate date," 
"ExposureKey string,"
"SoMBalance decimal(19,2),"
"SoMBucket int,"
"ActivityDate date,"
"EmployeeId string,"
"EmployeeName string,"
"TeamLeader STRING,"
"ActivityTypeKey string,"
"ActivityName string,"
"ActivityNameFull string,"
"BaseCategory string,"
"1stLevelCategory string,"
"2ndLevelCategory string,"
"3rdLevelCategory string,"
"ActivityGroup string,"
"NotificationsFlag string,"
"Hierarchy int,"
"IsRPC string,"
"GroupActionCode string,"
"ReportingCategory string,"
"DNCcategory string,"
"ActivityTypeKey_IRT string,"
"ActivityName_IRT string,"
"ActivityDate_IRT date,"
"ActivityUser_IRT string,"
"ActivityUserId_IRT string,"
"NewPipelineWithinMonth_IRT int,"
"ImplementedWithinMonth_IRT int,"
"InSpecificListOfActivities int,"
"AssignmentChannel STRING ,"
"InsertTimeStamp timestamp"
)

In [0]:
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"
delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date,  # type: ignore
)


##### Create Delta Tables with selected fields & Temporary Views for each delta table

In [0]:
# New Code
AgentInfo = spark.sql(
f'''
SELECT DISTINCT B.EmployeePartyKey
     , A.FullName
     , A.`3rdlevelAssignment`
     , A.TeamLeader
FROM `{env_prefix}-reportinputfiles`.repoint.core_assignmentlevelsmapping  A  
INNER JOIN `{env_prefix}-cdh`.primary.firmstructure B 
ON IF(LEN(TRIM(B.EmployeeUserName))=6,RIGHT(B.EmployeeUserName,5),B.EmployeeUserName)=IF(LEN(TRIM(A.`3rdlevelAssignment`))=6,RIGHT(A.`3rdlevelAssignment`,5),A.`3rdlevelAssignment`) 
AND B.EmployeeCategory = "Employee" 
AND B.SourceSystem = 'qca' 
AND B.BusinessDate = '{run_date}'
WHERE A.ReferenceDate = (SELECT MAX(ReferenceDate) FROM `{env_prefix}-reportinputfiles`.repoint.core_assignmentlevelsmapping WHERE ReferenceDate<='{run_date}') 
''')   
AgentInfo.createOrReplaceTempView("AgentInfo")

In [0]:
interaction = spark.sql(
f'''
select distinct ExposureKey
               ,InteractionDate
               ,CreatedBy
               ,InteractionTypeKey
               ,InteractionType
               ,CreatedDate  
from `{env_prefix}-cdh`.primary.get_interactions_by_business_date({run_date!r}) 
WHERE PortfolioName = 'Core Retail'
AND InteractionDate <= '{run_date}' AND InteractionDate > '{som_date}' 
''')   
interaction.createOrReplaceTempView("interaction")

In [0]:
AssignmentDataMapping = spark.sql(
f'''
SELECT DISTINCT ExposureKey , SoMBalanceAmountEUR, SoMSourceBucket, AssignmentChannel
FROM `{env_prefix}-cdh-delivery`.repoint.Core_AssignmentDataMapping
WHERE BusinessDate = '{run_date}'

''')   
AssignmentDataMapping.createOrReplaceTempView("AssignmentDataMapping")

In [0]:
RA_ModificationsProductionIRTActionCodes = spark.sql(
f'''
select  
 la.ExposureKey              
,dm.SoMBalanceAmountEUR               
,dm.SoMSourceBucket                 
,InteractionDate             
,ag.`3rdlevelAssignment`    
,ag.`FullName`              
,ag.TeamLeader
,la.InteractionTypeKey       
,la.InteractionType           
,m.ActivityName             
,m.BaseCategory            
,m.1stLevelCategory          
,m.2ndLevelCategory         
,m.3rdLevelCategory           
,m.ActivityGroup              
,m.NotificationsFlag          
,m.Hierarchy                  
,m.IsRPC                     
,m.GroupActionCode          
,m.ReportingCategory        
,p.DNCcategory               
,p.ActivityTypeKey_IRT       
,p.ActivityName_IRT           
,p.ActivityDate_IRT          
,p.ActivityUser_IRT           
,p.ActivityUserId_IRT        
,p.NewPipelineWithinMonth_IRT 
,p.ImplementedWithinMonth_IRT 
,CASE WHEN la.InteractionTypeKey in ('152','110','145','170','201','202','203','3016','486','499','500','501','514','516','518','521','524','525','528','529','531','533','534','536','537','546','547','571','572','573','577','578','620','621','622','623','624','780','781') THEN 1 ELSE 0 END AS InSpecificListOfActivities
,dm.AssignmentChannel
from interaction la  
inner join AgentInfo ag 
on la.CreatedBy = ag.EmployeePartyKey
inner join AssignmentDataMapping  dm   
on dm.ExposureKey = la.ExposureKey    
inner join `{env_prefix}-reportinputfiles`.repoint.core_contactactivitiesmapping m --`{env_prefix}-cdh-delivery`.repoint.get_latest_contact_activities_mapping m 
on CAST(m.ActivityCode AS STRING) = la.InteractionTypeKey and m.Team = 'IRT'     
left join `{env_prefix}-cdh-delivery`.repoint.core_modificationsproduction p --`{env_prefix}-cdh-delivery`.repoint.core_modifications_production_uat_for_IRT p
on  p.ExposureKey = la.ExposureKey and p.BusinessDate = '{run_date}' and p.FinalAssignmentChannel = 'IRT'                          
''')   
RA_ModificationsProductionIRTActionCodes.createOrReplaceTempView("RA_ModificationsProductionIRTActionCodes")

In [0]:
final_df = spark.sql(
    f'''
SELECT   --den thelw distinct giati xreiazontai ta duplicates logw twn IRTs
 CAST('{run_date}' AS date)                     BusinessDate
,CAST(ExposureKey AS STRING)                    ExposureKey
,CAST(SoMBalanceAmountEUR AS decimal(19,2))     SoMBalance
,CAST(SoMSourceBucket AS INT)                   SoMBucket
,CAST(InteractionDate AS DATE)                  ActivityDate
,CAST(`3rdlevelAssignment` AS STRING)           EmployeeId
,CAST(`FullName` AS STRING)                     EmployeeName
        ,CAST(TeamLeader AS STRING)                     TeamLeader
,CAST(InteractionTypeKey AS STRING)             ActivityTypeKey
,CAST(InteractionType AS STRING)                ActivityName
,CAST(ActivityName AS STRING)                   ActivityNameFull
,CAST(BaseCategory AS STRING)                   BaseCategory
,CAST(1stLevelCategory AS STRING)               1stLevelCategory
,CAST(2ndLevelCategory AS STRING)               2ndLevelCategory
,CAST(3rdLevelCategory AS STRING)               3rdLevelCategory
,CAST(ActivityGroup AS STRING)                  ActivityGroup
,CAST(NotificationsFlag AS STRING)              NotificationsFlag
,CAST(Hierarchy AS INT)                         Hierarchy
,CAST(IsRPC AS STRING)                          IsRPC
,CAST(GroupActionCode AS STRING)                GroupActionCode
,CAST(ReportingCategory AS STRING)              ReportingCategory
,CAST(DNCcategory AS STRING)                    DNCcategory
,CAST(ActivityTypeKey_IRT AS STRING)            ActivityTypeKey_IRT
,CAST(ActivityName_IRT AS STRING)               ActivityName_IRT
,CAST(ActivityDate_IRT AS DATE)                 ActivityDate_IRT
,CAST(ActivityUser_IRT AS STRING)               ActivityUser_IRT
,CAST(ActivityUserId_IRT AS STRING)             ActivityUserId_IRT
,CAST(NewPipelineWithinMonth_IRT AS INT)        NewPipelineWithinMonth_IRT
,CAST(ImplementedWithinMonth_IRT AS INT)        ImplementedWithinMonth_IRT
,CAST(InSpecificListOfActivities AS INT)        InSpecificListOfActivities
,CAST(AssignmentChannel AS STRING)              AssignmentChannel
,CAST(GETDATE() AS timestamp)                   InsertTimeStamp
FROM RA_ModificationsProductionIRTActionCodes
''')
final_df.createOrReplaceTempView("final_df")    

In [0]:
if final_df.rdd.isEmpty():
    raise ValueError(f"Core_ModificationsProductionIRTActionCodes: final DataFrame is empty - aborting write to {delivery_table_path} to avoid overwriting with no data")
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = []
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
ModificationsProductionIRTActionCodes = spark.sql(
f'''
select * from {delivery_table_path}
where BusinessDate = '{run_date}'
'''
)
ModificationsProductionIRTActionCodes.createOrReplaceTempView("ModificationsProductionIRTActionCodes")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/ModificationsProductionIRTActionCodes/"
file_name = "ModificationsProductionIRTActionCodes_" +run_date+ ".csv"
ModificationsProductionIRTActionCodes.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)